# sql-lm — Fine-Tuning v4

**Changes vs v3:**
- Context length 512 → 1024 (unlocks 73% of BIRD examples previously skipped)
- Fine-tune batch size 32 → 16 (O(T²) attention cost)
- Fine-tune dataset rebuilt at MAX_LEN=1024
- Starting checkpoint: `step_76500` (same pretrain base — testing RoPE extrapolation)

**Session resume:** if Colab disconnects, re-run cells 1–4 then use the **Resume** cell instead of the Run cell.

## 1 — Clone repo

In [ ]:
import os, shutil, subprocess, sys

if os.path.exists('/content/sql-lm'):
    shutil.rmtree('/content/sql-lm')

subprocess.run(
    ['git', 'clone', 'https://github.com/preetishreddy/sql-lm.git', '/content/sql-lm'],
    check=True
)

sys.path = [p for p in sys.path if 'sql-lm' not in p]
sys.path.insert(0, '/content/sql-lm')
print('scripts:', os.listdir('/content/sql-lm/scripts'))

## 2 — Mount Drive + install deps

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q -U flax optax orbax-checkpoint tokenizers

## 3 — Verify TPU + XLA cache

In [ ]:
import jax
print(f'JAX: {jax.__version__}  |  devices: {jax.devices()}')

jax.config.update('jax_compilation_cache_dir',
                  '/content/drive/MyDrive/sql-lm-data/xla-cache')
jax.config.update('jax_persistent_cache_min_entry_size_bytes', 0)
jax.config.update('jax_persistent_cache_min_compile_time_secs', 1)

## 4 — Copy data from Drive

In [ ]:
import shutil, os

# Tokenizer
os.makedirs('/content/tokenizer', exist_ok=True)
shutil.copy('/content/drive/MyDrive/sql-lm-data/tokenizer/tokenizer.json',
            '/content/tokenizer/tokenizer.json')
print('Tokenizer copied.')

# Pretrain checkpoint (starting point for v4 — same as v1/v2/v3)
os.makedirs('/content/checkpoints', exist_ok=True)
src = '/content/drive/MyDrive/sql-lm-data/checkpoints/step_76500'
dst = '/content/checkpoints/step_76500'
if not os.path.exists(dst):
    print('Copying step_76500 (~200MB)...')
    shutil.copytree(src, dst)
print('step_76500 ready.')

## 5 — Build fine-tune dataset at 1024 tokens

This replaces the v2/v3 dataset. MAX_LEN=1024 means BIRD examples up to 1024 tokens are now
included instead of skipped. Expect significantly more BIRD examples and a larger dataset.

Takes ~15 minutes. Run once and save to Drive — skip on resume.

In [ ]:
import os, shutil

DRIVE_DATA = '/content/drive/MyDrive/sql-lm-data/finetune_v4'
LOCAL_DATA = '/content/data/finetune'

if os.path.exists(os.path.join(DRIVE_DATA, 'manifest.json')):
    print('v4 dataset already on Drive — copying locally...')
    os.makedirs(LOCAL_DATA, exist_ok=True)
    shutil.copytree(DRIVE_DATA, LOCAL_DATA, dirs_exist_ok=True)
    print('Done.')
else:
    print('Building v4 dataset (MAX_LEN=1024)...')
    from scripts.finetune_data import build
    manifest = build('/content/tokenizer/tokenizer.json', LOCAL_DATA)

    # Save to Drive for resume
    os.makedirs(DRIVE_DATA, exist_ok=True)
    for f in ['train_tokens.npy', 'train_mask.npy',
              'val_tokens.npy',   'val_mask.npy', 'manifest.json']:
        shutil.copy(os.path.join(LOCAL_DATA, f), os.path.join(DRIVE_DATA, f))
    print('Dataset saved to Drive.')

## 6 — Sanity check

In [ ]:
import jax, jax.numpy as jnp, numpy as np, json
from tokenizers import Tokenizer
from scripts.model import SQLTransformer
from scripts.config import *
from scripts.finetune import FT_DROPOUT, FT_TOTAL_STEPS, FT_BATCH_SIZE

tok = Tokenizer.from_file('/content/tokenizer/tokenizer.json')
assert tok.get_vocab_size() == 12288

model = SQLTransformer(
    vocab_size=VOCAB_SIZE, hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS,
    num_heads=NUM_HEADS, head_dim=HEAD_DIM, intermediate_dim=INTERMEDIATE_DIM,
    context_length=CONTEXT_LENGTH, rope_base=ROPE_BASE,
    dtype=jnp.bfloat16, dropout_rate=FT_DROPOUT,
)
params = model.init(jax.random.PRNGKey(0),
                    jnp.ones((1, CONTEXT_LENGTH), dtype=jnp.int32))['params']
n = sum(x.size for x in jax.tree_util.tree_leaves(params))
assert 30_000_000 < n < 33_000_000, f'unexpected param count {n:,}'

tokens = np.load('/content/data/finetune/train_tokens.npy')
masks  = np.load('/content/data/finetune/train_mask.npy')
assert tokens.shape[1] == 1024, f'expected seq_len=1024, got {tokens.shape[1]}'

manifest = json.load(open('/content/data/finetune/manifest.json'))

print(f'Context length: {CONTEXT_LENGTH}')
print(f'Model:   {n:,} params  |  dropout={FT_DROPOUT}  |  steps={FT_TOTAL_STEPS}  |  batch={FT_BATCH_SIZE}')
print(f'Dataset: {len(tokens):,} train examples  |  avg mask={masks.mean():.3f}')
print(f'Sources: {json.dumps(manifest["sources"], indent=2)}')
print('All checks passed — ready to train.')

## 7 — Run fine-tuning (Session 1)

Runs 5,000 steps from `step_76500`. Checkpoints save to Drive every 500 steps.
Metrics written to `ft_metrics_v4.jsonl` (separate from v3) on Drive.

In [ ]:
from scripts.finetune import finetune
state = finetune(metrics_tag='ft_metrics_v4')

## 8 — Resume (Session 2+)

Re-run cells 1–4 after a disconnect, then run this cell.

In [ ]:
import shutil, os, glob

# Copy v4 dataset from Drive
DRIVE_DATA = '/content/drive/MyDrive/sql-lm-data/finetune_v4'
LOCAL_DATA = '/content/data/finetune'
os.makedirs(LOCAL_DATA, exist_ok=True)
shutil.copytree(DRIVE_DATA, LOCAL_DATA, dirs_exist_ok=True)
print('Dataset restored.')

# Copy latest ft checkpoint from Drive
ft_ckpts = sorted(glob.glob(
    '/content/drive/MyDrive/sql-lm-data/checkpoints/ft_step_*'))
if ft_ckpts:
    latest = ft_ckpts[-1]
    name   = os.path.basename(latest)
    dst    = f'/content/checkpoints/{name}'
    if not os.path.exists(dst):
        print(f'Copying {name} from Drive...')
        shutil.copytree(latest, dst)
    print(f'Resuming from {name}')
else:
    print('No ft checkpoint found — starting from step_76500.')

from scripts.finetune import finetune
state = finetune(metrics_tag='ft_metrics_v4')

## 9 — Plot training curve

In [ ]:
import json, matplotlib.pyplot as plt
from pathlib import Path

path = '/content/drive/MyDrive/sql-lm-data/ft_metrics_v4.jsonl'
records = [json.loads(l) for l in Path(path).read_text().splitlines() if l.strip()]

train_r = [(r['step'], r['loss'], r['grad_norm']) for r in records if r['kind'] == 'train']
val_r   = [(r['step'], r['loss'])                 for r in records if r['kind'] == 'val']

ts, tl, gn = zip(*train_r)
fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

ax[0].plot(ts, tl, label='train', linewidth=0.9)
if val_r:
    vs, vl = zip(*val_r)
    ax[0].plot(vs, vl, 'o-', label='val', markersize=4)
    best_step = vs[vl.index(min(vl))]
    ax[0].axvline(best_step, color='green', linestyle='--', alpha=0.5,
                  label=f'best val @ step {best_step}')
ax[0].set_ylabel('loss'); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[0].set_title('sql-lm v4 fine-tuning (context=1024)')

ax[1].plot(ts, gn, color='orange', linewidth=0.9)
ax[1].axhline(1.0, color='red', linestyle='--', alpha=0.5, label='clip=1.0')
ax[1].set_ylabel('grad norm'); ax[1].set_xlabel('step')
ax[1].legend(); ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/ft_v4_curve.png', dpi=150)
plt.show()

if val_r:
    print(f'Best val loss: {min(vl):.4f} at step {best_step}')
print(f'Latest step: {ts[-1]}  |  train loss: {tl[-1]:.4f}')

## 10 — Evaluate on gretelai test set

Update `CKPT` to the best checkpoint after training completes.

In [ ]:
import orbax.checkpoint as ocp
import jax, jax.numpy as jnp
from tokenizers import Tokenizer
from scripts.model import SQLTransformer
from scripts.config import *
from scripts.eval_sql import evaluate_gretelai
from scripts.finetune import create_ft_optimizer, create_ft_schedule, FT_DROPOUT

CKPT = '/content/drive/MyDrive/sql-lm-data/checkpoints/ft_step_04500'  # update after training

model = SQLTransformer(
    vocab_size=VOCAB_SIZE, hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS,
    num_heads=NUM_HEADS, head_dim=HEAD_DIM, intermediate_dim=INTERMEDIATE_DIM,
    context_length=CONTEXT_LENGTH, rope_base=ROPE_BASE,
    dtype=jnp.bfloat16, dropout_rate=FT_DROPOUT,
)
dummy           = jnp.ones((1, CONTEXT_LENGTH), dtype=jnp.int32)
template_params = model.init(jax.random.PRNGKey(0), dummy)['params']
opt_template    = create_ft_optimizer(create_ft_schedule()).init(template_params)

ckptr    = ocp.PyTreeCheckpointer()
restored = ckptr.restore(CKPT, item={
    'params': template_params, 'opt_state': opt_template, 'step': 0})
params = restored['params']

tok = Tokenizer.from_file('/content/tokenizer/tokenizer.json')
print(f'Loaded: {CKPT}\n')

results_v4 = evaluate_gretelai(params, model, tok)